In [1]:
pip install selenium webdriver-manager


Note: you may need to restart the kernel to use updated packages.


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import time
import re

In [3]:
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)
actions = ActionChains(driver)


In [4]:
target_url = "https://www.bumeran.com.pe/en-lima/empleos-area-tecnologia-sistemas-y-telecomunicaciones-subarea-programacion-full-time-publicacion-menor-a-15-dias.html"
driver.get(target_url)
time.sleep(5)  # Espera inicial para carga completa


In [5]:
def extraer_enlaces_pagina():
    """Extrae enlaces de empleos de la página actual"""
    try:
        page_source = driver.page_source
        patron_enlaces = re.compile(r'href="(/empleo/[^"]+|/empleos/[^"]+)"')
        enlaces_encontrados = patron_enlaces.findall(page_source)
        
        enlaces_unicos = []
        base_url = "https://www.bumeran.com.pe"
        for enlace in enlaces_encontrados:
            enlace_completo = f"{base_url}{enlace}" if enlace.startswith("/") else enlace
            if enlace_completo not in enlaces_unicos:
                enlaces_unicos.append(enlace_completo)
        
        return enlaces_unicos
    except Exception as e:
        return []

def navegar_siguiente_pagina(pagina_actual):
    """Navega a la siguiente página"""
    try:
        siguiente_pagina = pagina_actual + 1
        nueva_url = f"{target_url}?page={siguiente_pagina}"
        driver.get(nueva_url)
        time.sleep(3)
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//body")))
        return True
    except:
        return False

def hacer_scroll_completo():
    """Hace scroll completo de la página"""
    driver.execute_script("window.scrollTo({top: document.body.scrollHeight, behavior: 'smooth'});")
    time.sleep(3)

# Proceso principal (silencioso)
todos_enlaces = []
pagina_actual = 1
max_paginas = 6

while pagina_actual <= max_paginas:
    hacer_scroll_completo()
    enlaces_pagina = extraer_enlaces_pagina()
    
    if enlaces_pagina:
        nuevos_enlaces = [link for link in enlaces_pagina if link not in todos_enlaces]
        todos_enlaces.extend(nuevos_enlaces)
    
    if pagina_actual < max_paginas:
        if not navegar_siguiente_pagina(pagina_actual):
            break
    
    pagina_actual += 1

# Mostrar solo los resultados finales
print("\nLista completa de enlaces encontrados:\n")
for idx, url in enumerate(todos_enlaces, 1):
    print(f"{idx}. {url}")

print(f"\nTotal de enlaces: {len(todos_enlaces)}")
driver.quit()


Lista completa de enlaces encontrados:

1. https://www.bumeran.com.pe/empleos/desarrollador-senior-java-ntt-data-1116752169.html
2. https://www.bumeran.com.pe/empleos/software-architect-talent-house-peru-s.a.c.-1116750633.html
3. https://www.bumeran.com.pe/empleos/analista-programador-power-builder-junior-1116745817.html
4. https://www.bumeran.com.pe/empleos/desarrollador-front-ios-summit-s.a.c-1116742593.html
5. https://www.bumeran.com.pe/empleos/analista-programador-de-tecnologias-de-la-informacion-instituto-quimioterapico-s.a.-iqfarma-1116740821.html
6. https://www.bumeran.com.pe/empleos/frontend-developer-react-maseka-consulting-s.a.c.-1116739942.html
7. https://www.bumeran.com.pe/empleos/analista-programador-de-sistemas-hibrido-tecsup-1116737470.html
8. https://www.bumeran.com.pe/empleos/data-scientist-associate-|-desarrollo-de-modelos-riesgos-bbva-1116752543.html
9. https://www.bumeran.com.pe/empleos/frontend-developer-react-manpowergroup-peru-1116752520.html
10. https://www.bum

In [6]:
import csv
import time
import random
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException

def configurar_driver():
    """Configura el driver con opciones para evitar detección"""
    chrome_options = Options()
    
    # Configuraciones para evitar detección
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    chrome_options.add_argument("--start-maximized")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    
    # Modo headless opcional (más rápido pero más detectable)
    # chrome_options.add_argument("--headless=new")
    
    driver = webdriver.Chrome(service=service, options=chrome_options)
    
    # Eliminar huellas de automatización
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": """
            Object.defineProperty(navigator, 'webdriver', {
                get: () => undefined
            });
            window.navigator.chrome = {
                runtime: {},
            };
        """
    })
    
    return driver

def extraer_detalles_trabajo(driver, url):
    """Extrae detalles con múltiples estrategias de respaldo"""
    detalles = {
        "titulo": "No especificado",
        "ubicacion": "No especificado",
        "modalidad": "No especificado",
        "descripcion": "No especificado",
        "url": url
    }
    
    try:
        # Navegar con comportamiento humano simulado
        driver.get(url)
        time.sleep(random.uniform(2.0, 4.0))  # Espera aleatoria
        
        # Scroll para activar posibles cargas dinámicas
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight/3);")
        time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
        time.sleep(1)
        
        # 1. Título - Múltiples estrategias
        try:
            detalles["titulo"] = driver.find_element(By.CSS_SELECTOR, "h1.sc-hMkerz").text.strip()
        except NoSuchElementException:
            try:
                detalles["titulo"] = driver.find_element(By.XPATH, '//h1[contains(@class, "job-title")]').text.strip()
            except:
                pass
        
        # 2. Ubicación - Múltiples estrategias
        try:
            detalles["ubicacion"] = driver.find_element(By.CSS_SELECTOR, "h2.sc-fjmucm").text.strip()
        except NoSuchElementException:
            try:
                detalles["ubicacion"] = driver.find_element(By.XPATH, '//*[contains(text(), "Lima") or contains(text(), "Peru")]').text.strip()
            except:
                pass
        
        # 3. Modalidad - Múltiples estrategias
        try:
            detalles["modalidad"] = driver.find_element(By.CSS_SELECTOR, "p.sc-eBOlBG").text.strip()
        except NoSuchElementException:
            try:
                modalidad = driver.find_element(By.XPATH, '//*[contains(text(), "Remoto") or contains(text(), "Presencial") or contains(text(), "Híbrido")]')
                detalles["modalidad"] = modalidad.text.strip()
            except:
                pass
        
        # 4. Descripción - Múltiples estrategias
        try:
            desc_element = driver.find_element(By.CSS_SELECTOR, "p.sc-fkNhrH")
            detalles["descripcion"] = desc_element.get_attribute('innerHTML').strip()
        except NoSuchElementException:
            try:
                desc_element = driver.find_element(By.XPATH, '//div[contains(@class, "job-description")]')
                detalles["descripcion"] = desc_element.text.strip()
            except:
                pass
        
    except Exception as e:
        print(f"Error procesando {url}: {str(e)[:100]}...")  # Limitar longitud del mensaje de error
    
    return detalles

def guardar_en_csv(enlaces, filename="ofertas_trabajo.csv"):
    """Ejecuta el scraping con gestión profesional de errores"""
    driver = configurar_driver()
    
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=[
            "titulo", "ubicacion", "modalidad", "descripcion", "url"
        ])
        writer.writeheader()
        
        for idx, url in enumerate(enlaces, 1):
            try:
                print(f"[{idx}/{len(enlaces)}] Procesando: {url[:60]}...")
                detalles = extraer_detalles_trabajo(driver, url)
                writer.writerow(detalles)
                
                # Pausa variable con comportamiento humano
                wait_time = random.uniform(3.0, 7.0)
                time.sleep(wait_time)
                
                # Rotar User-Agent periódicamente
                if idx % 5 == 0:
                    driver.execute_cdp_cmd('Network.setUserAgentOverride', {
                        "userAgent": f"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/{random.randint(90,115)}.0.0.0 Safari/537.36"
                    })
                
            except Exception as e:
                print(f"Error grave procesando URL {idx}: {str(e)[:100]}...")
                continue
    
    driver.quit()
    print(f"\n✅ ¡Proceso completado! Resultados en {filename}")

# Ejecución principal con manejo de errores global
try:
    guardar_en_csv(todos_enlaces)
except Exception as e:
    print(f"Error inesperado: {str(e)}")
finally:
    print("Proceso finalizado")

[1/114] Procesando: https://www.bumeran.com.pe/empleos/desarrollador-senior-java...
[2/114] Procesando: https://www.bumeran.com.pe/empleos/software-architect-talent...
[3/114] Procesando: https://www.bumeran.com.pe/empleos/analista-programador-powe...
[4/114] Procesando: https://www.bumeran.com.pe/empleos/desarrollador-front-ios-s...
[5/114] Procesando: https://www.bumeran.com.pe/empleos/analista-programador-de-t...
[6/114] Procesando: https://www.bumeran.com.pe/empleos/frontend-developer-react-...
[7/114] Procesando: https://www.bumeran.com.pe/empleos/analista-programador-de-s...
[8/114] Procesando: https://www.bumeran.com.pe/empleos/data-scientist-associate-...
[9/114] Procesando: https://www.bumeran.com.pe/empleos/frontend-developer-react-...
[10/114] Procesando: https://www.bumeran.com.pe/empleos/backend-developer-nodejs-...
[11/114] Procesando: https://www.bumeran.com.pe/empleos/analista-programador-de-s...
[12/114] Procesando: https://www.bumeran.com.pe/empleos/jefe-de-sistemas-t